# The Plugin System

A `Plugin` pairs a `TableParser` (reads tabular data) with a `MetadataParser` (reads file metadata).
The `PLUGINS` dict maps plugin IDs to `Plugin` instances for all built-in formats.

Auto-detection runs in three stages:
1. **`detect_from_ext`** – filters by file extension
2. **`detect_from_metadata`** – filters by magic token matching
3. **`detect_from_columns`** – picks the highest-scoring normalizer match

The top-level `detect()` orchestrates all three stages and returns early when unambiguous.

In [ ]:
import re
from pathlib import Path

from bdf.plugins import Plugin, PLUGINS, detect, detect_from_ext, detect_from_metadata, detect_from_columns
from bdf.table_parsers import DelimTxtParser
from bdf.normalizers import TableNormalizer, Syn
from bdf.metadata_parsers import TxtPreambleParser, MetadataSchema

In [ ]:
# Each key is a plugin ID; the value is the Plugin instance
list(PLUGINS.keys())

## Constructing a custom `Plugin`

A `Plugin` takes a `table_parser` and a `metadata_parser`. The structure mirrors the built-in
`BIOLOGIC_MPT` plugin: a `DelimTxtParser` for the tabular data and a `TxtPreambleParser` for the header metadata.

In [ ]:
custom_plugin = Plugin(
    table_parser=DelimTxtParser(
        normalizer=TableNormalizer(
            test_time_second=(Syn(root="time/{unit}"),),
            voltage_volt=(Syn(root="ecell/{unit}"),),
            current_ampere=(Syn(root="i/{unit}"),),
        )
    ),
    metadata_parser=TxtPreambleParser(
        magic=("BT-Lab ASCII FILE",),
        regex_patterns=MetadataSchema(
            start_time=re.compile(r"Acquisition started on\s*:\s*(.+)"),
        ),
    ),
)
custom_plugin

## Stage 1 — `detect_from_ext()`

Extension filtering is cheap but imprecise: `.txt` is used by multiple formats.
`detect_from_ext()` returns all plugins that accept the file's extension.

In [ ]:
biologic_file = Path("../tests/data/biologic/Sample_data_biologic_01_MB_CA1.txt")

ext_candidates = detect_from_ext(biologic_file)
print("Candidates from extension (.txt):", list(ext_candidates.keys()))

## Stage 2 — `detect_from_metadata()`

Magic token matching reads only the file's head bytes and is much more precise.
`detect_from_metadata()` filters the candidates to those whose `metadata_parser` matches.

In [ ]:
meta_candidates = detect_from_metadata(biologic_file, ext_candidates)
print("Candidates after metadata filter:", list(meta_candidates.keys()))

## Stage 3 — `detect_from_columns()`

`detect_from_columns()` sniffs the column headings and returns the plugin with the highest normalizer score.
It returns a `(plugin_id, Plugin)` tuple.

In [ ]:
plugin_id, plugin = detect_from_columns(biologic_file, meta_candidates)
print("Resolved plugin:", plugin_id)

## Top-level `detect()` shortcut

`detect()` orchestrates all three stages and returns early when the candidates are already unambiguous.

In [ ]:
plugin_id, plugin = detect(biologic_file)
print("Detected:", plugin_id)

In [ ]:
# Read the normalised data using the detected plugin
plugin.table_parser.read(biologic_file).collect()